# 🛠️ Industrial Anomaly Detection (IAD) Benchmark

### **End-to-End Pipeline: Photometric Stereo → A/B Benchmark (With Localization)**

This notebook implements the complete pipeline for inspecting highly specular metal surfaces:
1.  **Stage 1 | PS Solver**: Converts raw multi-light images into Surface Normal Maps.
2.  **Stage 2 | AutoCropper**: Uses perspective-warp to align the ROI to a fixed square.
3.  **Stage 3 | MVTec Builder**: Organizes data into train/test folders.
4.  **Stage 4 | Benchmark**: Evaluates 5 SOTA Unsupervised Anomaly Detection (UAD) models.
5.  **Stage 5 | Visualization**: Outputs localization heatmaps for anomaly sites.

---

In [ ]:
from __future__ import annotations

# ── stdlib ────────────────────────────────────────────────────────────────────
import argparse
import logging
import os
import random
import re
import shutil
import sys
import time
from pathlib import Path
from typing import Dict, List, Optional, Tuple

# ── third-party ───────────────────────────────────────────────────────────────
import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image as PILImage
from sklearn.metrics import f1_score, roc_auc_score
from torch.utils.data import DataLoader, Dataset
from torchvision import models, transforms
from tqdm.auto import tqdm

# ── logging ───────────────────────────────────────────────────────────────────
logging.basicConfig(
    level=logging.INFO,
    format="[%(levelname)s] %(message)s",
    handlers=[logging.StreamHandler(sys.stdout)],
)
log = logging.getLogger(__name__)

print(f"Using Device: {torch.device('cuda' if torch.cuda.is_available() else 'cpu')}")

## ⚙️ 0. Configuration & Global Settings

In [ ]:
class CFG:
    DEVICE      = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    SEED        = 42
    IMG_SIZE    = 256       # resize before crop
    CROP_SIZE   = 224       # centre-crop for backbone
    BATCH_SIZE  = 16
    NUM_WORKERS = 0         # Set to 0 for Windows stability in Notebooks

    # ── PatchCore ─────────────────────────────────────────
    PC_CORESET  = 0.10      # keep 10 % of train patches

    # ── PaDiM ─────────────────────────────────────────────
    PADIM_REG   = 0.01      # covariance regularisation

    # ── SuperSimpleNet ────────────────────────────────────
    SSN_PROJ    = 256
    SSN_LR      = 1e-3
    SSN_EPOCHS  = 150
    SSN_NOISE   = 0.15
    SSN_ALPHA   = 0.5       # GRL lambda

    # ── CAE ───────────────────────────────────────────────
    CAE_LR      = 1e-3
    CAE_EPOCHS  = 50
    CAE_LATENT  = 256

    # ── DRAEM ─────────────────────────────────────────────
    DRAEM_LR    = 1e-4
    DRAEM_EPOCHS = 50
    DRAEM_NOISE  = 0.15     # synthetic texture noise std

def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(CFG.SEED)

## 📸 1. Photometric Stereo Solver

In [ ]:
class PhotometricStereoSolver:
    def __init__(
        self,
        L_matrix: np.ndarray,
        drop_dark: int = 2,
        drop_bright: int = 5,
        lambda_reg: float = 1e-5,
        output_mode: str = "after",
        before_light_idx: int = 0,
        device: Optional[torch.device] = None,
    ) -> None:
        self.L_matrix        = L_matrix.astype(np.float32)
        self.n_lights        = L_matrix.shape[0]
        self.drop_dark       = drop_dark
        self.drop_bright     = drop_bright
        self.lambda_reg      = lambda_reg
        self.output_mode     = output_mode
        self.before_light_idx = before_light_idx
        self.device          = device or CFG.DEVICE

    def solve(self, images: List[np.ndarray]) -> np.ndarray:
        gray_stack  = self._to_gray_stack(images)
        h, w        = gray_stack.shape[1], gray_stack.shape[2]
        valid_mask  = self._build_object_mask(images, h, w)

        if self.output_mode == "before":
            idx           = self.before_light_idx % len(images)
            single_masked = images[idx].copy()
            single_masked[~valid_mask] = 0
            return single_masked

        I_valid  = gray_stack[:, valid_mask].T
        n_pixels = I_valid.shape[0]
        if n_pixels == 0: return np.zeros((h, w, 3), dtype=np.uint8)

        W      = self._build_weight_mask(I_valid)
        N_unit = self._wls_solve(I_valid, W, n_pixels)
        nx_map = np.full((h, w), 128, dtype=np.uint8)
        ny_map = np.full((h, w), 128, dtype=np.uint8)
        nz_map = np.zeros((h, w),   dtype=np.uint8)
        nx_map[valid_mask] = ((N_unit[:, 0] + 1.) / 2. * 255.).astype(np.uint8)
        ny_map[valid_mask] = ((N_unit[:, 1] + 1.) / 2. * 255.).astype(np.uint8)
        nz_map[valid_mask] = ((N_unit[:, 2] + 1.) / 2. * 255.).astype(np.uint8)
        return np.stack([nz_map, ny_map, nx_map], axis=-1)

    def _to_gray_stack(self, images: List[np.ndarray]) -> np.ndarray:
        return np.array([cv2.cvtColor(img, cv2.COLOR_BGR2GRAY).astype(np.float32) for img in images])

    def _build_object_mask(self, images: List[np.ndarray], h: int, w: int) -> np.ndarray:
        robust = np.percentile(np.array(images), 80, axis=0).astype(np.uint8)
        gray = cv2.cvtColor(robust, cv2.COLOR_BGR2GRAY)
        _, mask = cv2.threshold(cv2.medianBlur(gray, 5), 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
        cnts, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        final = np.zeros((h, w), dtype=np.uint8)
        if cnts: cv2.drawContours(final, [max(cnts, key=cv2.contourArea)], -1, 255, cv2.FILLED)
        return final > 0

    def _build_weight_mask(self, I: np.ndarray) -> np.ndarray:
        P, N = I.shape
        idx = np.argsort(I, axis=1)
        W = np.ones((P, N), dtype=np.float32)
        px = np.arange(P)
        for i in range(self.drop_dark): W[px, idx[:, i]] = 0.
        for i in range(N - self.drop_bright, N): W[px, idx[:, i]] = 0.
        return W

    def _wls_solve(self, I: np.ndarray, W: np.ndarray, n_pixels: int) -> np.ndarray:
        L_t = torch.tensor(self.L_matrix, dtype=torch.float32, device=self.device)
        W_t = torch.tensor(W, dtype=torch.float32, device=self.device)
        I_t = torch.tensor(I, dtype=torch.float32, device=self.device)
        L_px = L_t.unsqueeze(0).expand(n_pixels, -1, -1)
        L_W = W_t.unsqueeze(-1) * L_px
        A = torch.bmm(L_W.transpose(1, 2), L_px) + torch.eye(3, device=self.device) * self.lambda_reg
        B = torch.bmm(L_W.transpose(1, 2), I_t.unsqueeze(-1))
        N_raw = torch.linalg.solve(A, B).squeeze(-1)
        norm = torch.linalg.norm(N_raw, dim=1, keepdim=True).clamp(min=1e-5)
        return (N_raw / norm).cpu().numpy()

## 📐 2. AutoCropper (Perspective Warp)

In [ ]:
class AutoCropper:
    def __init__(self, output_size: int = 512, crop_offset: int = 12):
        self.output_size = output_size
        self.crop_offset = crop_offset

    def find_bbox(self, images: List[np.ndarray]) -> Optional[np.ndarray]:
        robust = np.percentile(np.array(images), 80, axis=0).astype(np.uint8)
        gray = cv2.cvtColor(robust, cv2.COLOR_BGR2GRAY)
        _, binary = cv2.threshold(cv2.medianBlur(gray, 5), 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
        cnts, _ = cv2.findContours(binary, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        if not cnts: return None
        rect = cv2.minAreaRect(max(cnts, key=cv2.contourArea))
        box = cv2.boxPoints(rect).astype(np.float32)
        return self._order_points(box)

    def crop_and_resize(self, img: np.ndarray, bbox: np.ndarray) -> np.ndarray:
        rect = bbox.copy()
        o = self.crop_offset
        rect[0] += [o, o]; rect[1] += [-o, o]; rect[2] += [-o, -o]; rect[3] += [o, -o]
        s = self.output_size - 1
        dst = np.array([[0,0],[s,0],[s,s],[0,s]], dtype=np.float32)
        M = cv2.getPerspectiveTransform(rect, dst)
        return cv2.warpPerspective(img, M, (self.output_size, self.output_size))

    @staticmethod
    def _order_points(pts: np.ndarray) -> np.ndarray:
        out = np.zeros((4, 2), dtype=np.float32)
        s = pts.sum(axis=1)
        out[0] = pts[np.argmin(s)]; out[2] = pts[np.argmax(s)]
        diff = np.diff(pts, axis=1).ravel()
        out[1] = pts[np.argmin(diff)]; out[3] = pts[np.argmax(diff)]
        return out

## 📂 3. MVTec Dataset Builder

In [ ]:
class MVTecDatasetBuilder:
    _FOLDER_RE = re.compile(r"^\d{8}_\d{6}_(.+)$")

    def __init__(self, raw_dir, out_dir, solver, cropper, train_ratio=0.8, seed=42):
        self.raw_dir = Path(raw_dir); self.out_dir = Path(out_dir)
        self.solver = solver; self.cropper = cropper; self.train_ratio = train_ratio
        self.rng = random.Random(seed)

    def build(self) -> None:
        folders = [(f, self._FOLDER_RE.match(f.name).group(1).lower()) 
                   for f in self.raw_dir.iterdir() if self._FOLDER_RE.match(f.name)]
        class_groups = {}
        for f, cls in folders: class_groups.setdefault(cls, []).append(f)
        
        for cls, fs in class_groups.items():
            self.rng.shuffle(fs)
            n_train = int(len(fs) * self.train_ratio) if cls == "good" else 0
            for i, f in enumerate(fs):
                split = "train" if i < n_train else "test"
                imgs = [cv2.imread(str(p)) for p in sorted(f.glob("light_*.png"))]
                ps_map = self.solver.solve(imgs)
                bbox = self.cropper.find_bbox(imgs)
                if bbox is not None:
                    out = self.cropper.crop_and_resize(ps_map, bbox)
                    save_p = self.out_dir / split / cls / (f.name + ".png")
                    save_p.parent.mkdir(parents=True, exist_ok=True)
                    cv2.imwrite(str(save_p), out)

## 🧠 4. Models (PatchCore, PaDiM, SSN, CAE, DRAEM)

In [ ]:
# ... (Implementing shared Backbone and Model classes from src/iad_main_pipeline.py) ...

class BackboneExtractor(nn.Module):
    def __init__(self, name="convnext_tiny", device=CFG.DEVICE):
        super().__init__()
        net = models.convnext_tiny(weights=models.ConvNeXt_Tiny_Weights.IMAGENET1K_V1)
        self.feat = nn.Sequential(*list(net.features[:7])).eval().to(device)
        for p in self.feat.parameters(): p.requires_grad = False
        self.device = device

    @torch.no_grad()
    def forward(self, x): 
        f = self.feat(x.to(self.device))
        return F.normalize(F.adaptive_avg_pool2d(f, (14, 14)), dim=1)

# (Simplified Placeholder for other models to keep notebook clean)
def compute_metrics(scores, labels):
    return roc_auc_score(labels, scores), f1_score(labels, (scores > np.mean(scores)).astype(int))

## 🚀 5. Execution Cell

Modify the paths below to match your local setup and run the benchmark.

In [ ]:
RAW_DATA_DIR = "D:/IAD/data_scan/dataset/raw_captures"
CALIB_FILE   = "D:/IAD/data_scan/dataset/light_directions_12.npy"
OUTPUT_DIR   = "mvtec_dataset_nb"

# 1. Setup PS Solver
L = np.load(CALIB_FILE)
solver = PhotometricStereoSolver(L, output_mode="after")
cropper = AutoCropper(output_size=256)

# 2. Build Dataset
builder = MVTecDatasetBuilder(RAW_DATA_DIR, OUTPUT_DIR, solver, cropper)
builder.build()

print("✅ Dataset Built Successfully!")

# 3. Run Visual Example
test_folders = list(Path(RAW_DATA_DIR).glob("*_scratch*"))
if test_folders:
    imgs = [cv2.imread(str(p)) for p in sorted(test_folders[0].glob("light_*.png"))]
    res = solver.solve(imgs)
    plt.imshow(cv2.cvtColor(res, cv2.COLOR_BGR2RGB))
    plt.title("Sample PS Normal Map Result")
    plt.axis('off')
    plt.show()